# Coding Assignment 2: Advanced Ensemble Learning and Evaluation for Cancer Prediction

# ID 2671508 _ Tasnuba Tasnim

## Phase A: Data Preparation and Feature Extraction

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

# Load the Breast Cancer Wisconsin dataset
cancer_dataset = load_breast_cancer()

# Transform into a Pandas DataFrame for easier manipulation
data_frame = pd.DataFrame(data=cancer_dataset.data, columns=cancer_dataset.feature_names)
data_frame['target_class'] = cancer_dataset.target

# Display initial entries and data structure to confirm loading
display(data_frame.head())
print(data_frame.info())

### Inspecting Data Integrity: Missing Value Assessment

In [ ]:
# Identify any columns with null entries
null_counts = data_frame.isnull().sum()
null_counts = null_counts[null_counts > 0]

# Report findings on missing values
if not null_counts.empty:
    print("Columns with identified missing values:")
    print(null_counts)
else:
    print("No missing values detected across the dataset.")

### Feature Selection: Identifying Key Predictors (Top 5 Correlations)

In [ ]:
# Compute absolute correlations of features with the target variable
target_correlations = data_frame.corr()['target_class'].abs().sort_values(ascending=False)

# Extract the top 5 features, excluding the target_class itself
selected_features = target_correlations[1:6].index.tolist()

print(f"The five most correlated features with the target variable are: {selected_features}")

# Create feature matrix (X) and target vector (y) using these selected features
feature_matrix = data_frame[selected_features]
target_vector = data_frame['target_class']

### Feature Transformation: StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler

data_scaler = StandardScaler()

scaled_features = data_scaler.fit_transform(feature_matrix)

# Convert scaled features back to a DataFrame
scaled_features_df = pd.DataFrame(scaled_features, columns=feature_matrix.columns)

print("First 5 rows of scaled features:")
display(scaled_features_df.head())
print(f"Average mean of scaled features (expected near 0): {scaled_features_df.mean().mean():.2f}")
print(f"Average standard deviation of scaled features (expected near 1): {scaled_features_df.std().mean():.2f}")

## Phase B: Predictive Model Development and Optimization

### 1. Baseline Model: Decision Tree Classifier

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

# Divide data into training and testing subsets
train_features, test_features, train_targets, test_targets = train_test_split(
    scaled_features, target_vector, test_size=0.3, random_state=42
)

# Initialize and train the Decision Tree model
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(train_features, train_targets)

# Generate predictions and probabilities on the test set
tree_predictions = tree_model.predict(test_features)
tree_probabilities = tree_model.predict_proba(test_features)[:, 1]

# Evaluate the Decision Tree model's performance
tree_accuracy = accuracy_score(test_targets, tree_predictions)
tree_f1_score = f1_score(test_targets, tree_predictions)
tree_roc_auc = roc_auc_score(test_targets, tree_probabilities)

print(f"Decision Tree Model Accuracy: {tree_accuracy:.4f}")
print(f"Decision Tree Model F1-Score: {tree_f1_score:.4f}")
print(f"Decision Tree Model ROC-AUC Score: {tree_roc_auc:.4f}")

### 2. Advanced Ensemble: Gradient Boosting Classifier with Hyperparameter Tuning

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

# Initialize a Gradient Boosting Classifier instance
gbm_model = GradientBoostingClassifier(random_state=42)

# Define the hyperparameter search space for GridSearchCV
gbm_param_grid = {
    'n_estimators': [50, 100, 200],  # Number of boosting stages
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

# Configure GridSearchCV for hyperparameter optimization
gbm_grid_search = GridSearchCV(
    estimator=gbm_model,
    param_grid=gbm_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# Execute GridSearchCV on the training data
gbm_grid_search.fit(train_features, train_targets)

# Retrieve the best performing model from the search
best_gbm_model = gbm_grid_search.best_estimator_

print(f"Optimal Hyperparameters for Gradient Boosting: {gbm_grid_search.best_params_}")

# Make predictions and calculate probabilities using the optimized model
gbm_predictions = best_gbm_model.predict(test_features)
gbm_probabilities = best_gbm_model.predict_proba(test_features)[:, 1]

# Evaluate the optimized Gradient Boosting model
gbm_accuracy = accuracy_score(test_targets, gbm_predictions)
gbm_f1_score = f1_score(test_targets, gbm_predictions)
gbm_roc_auc = roc_auc_score(test_targets, gbm_probabilities)

print(f"Gradient Boosting Model Accuracy: {gbm_accuracy:.4f}")
print(f"Gradient Boosting Model F1-Score: {gbm_f1_score:.4f}")
print(f"Gradient Boosting Model ROC-AUC Score: {gbm_roc_auc:.4f}")

### 3. Support Vector Machine (SVM) with Non-linear Kernel and Hyperparameter Tuning

In [ ]:
from sklearn.svm import SVC

# Initialize the Support Vector Classifier
# probability=True is needed for ROC-AUC scoring
svm_model = SVC(random_state=42, probability=True)

# Define the hyperparameter grid for SVM's GridSearchCV
svm_param_grid = {
    'C': [0.1, 1, 10, 100],  # Regularization strength
    'gamma': [0.001, 0.01, 0.1, 1], # Kernel coefficient for 'rbf'
    'kernel': ['rbf'] # Using a non-linear Radial Basis Function (RBF) kernel
}

# Configure GridSearchCV for SVM hyperparameter tuning
svm_grid_search = GridSearchCV(
    estimator=svm_model,
    param_grid=svm_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# Fit GridSearchCV on the training data to find optimal SVM parameters
svm_grid_search.fit(train_features, train_targets)

# Get the best SVM model found by the grid search
best_svm_model = svm_grid_search.best_estimator_

print(f"Optimal Hyperparameters for SVM: {svm_grid_search.best_params_}")

# Generate predictions and probabilities with the best SVM model
svm_predictions = best_svm_model.predict(test_features)
svm_probabilities = best_svm_model.predict_proba(test_features)[:, 1]

# Evaluate the performance of the best SVM model
svm_accuracy = accuracy_score(test_targets, svm_predictions)
svm_f1_score = f1_score(test_targets, svm_predictions)
svm_roc_auc = roc_auc_score(test_targets, svm_probabilities)

print(f"SVM Model Accuracy: {svm_accuracy:.4f}")
print(f"SVM Model F1-Score: {svm_f1_score:.4f}")
print(f"SVM Model ROC-AUC Score: {svm_roc_auc:.4f}")

## Phase C: Visualizing Model Performance and Advanced Evaluation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix


# 1. Hyperparameter Impact Plot: Gradient Boosting

# Use the n_estimators values from the GridSearchCV parameter grid
estimators_options = sorted(gbm_param_grid['n_estimators'])

training_accuracy_scores = []
validation_accuracy_scores = []

# Retain the best learning_rate and max_depth found by GridSearchCV
best_learning_rate = gbm_grid_search.best_params_['learning_rate']
best_max_depth = gbm_grid_search.best_params_['max_depth']

for estimator_count in estimators_options:

    # Create a Gradient Boosting model for each n_estimators value
    impact_model = GradientBoostingClassifier(
        n_estimators=estimator_count,
        learning_rate=best_learning_rate,
        max_depth=best_max_depth,
        random_state=42
    )

    # Train the model using the training dataset
    impact_model.fit(train_features, train_targets)

    # Calculate training accuracy
    training_accuracy = impact_model.score(
        train_features,
        train_targets
    )

    # Calculate mean validation accuracy using 5-fold cross-validation
    cross_validation_scores = cross_val_score(
        impact_model,
        train_features,
        train_targets,
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )

    mean_validation_accuracy = cross_validation_scores.mean()

    training_accuracy_scores.append(training_accuracy)
    validation_accuracy_scores.append(mean_validation_accuracy)


# Generate the hyperparameter impact plot
plt.figure(figsize=(10, 6))

plt.plot(
    estimators_options,
    training_accuracy_scores,
    marker='o',
    linewidth=2,
    label='Training Accuracy'
)

plt.plot(
    estimators_options,
    validation_accuracy_scores,
    marker='s',
    linewidth=2,
    label='Validation Accuracy'
)

plt.title(
    'Impact of Number of Estimators on Gradient Boosting Accuracy',
    fontsize=14,
    fontweight='bold'
)

plt.xlabel('Number of Estimators', fontsize=12)
plt.ylabel('Accuracy Score', fontsize=12)
plt.xticks(estimators_options)
plt.ylim(0.80, 1.02)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Save the plot before displaying it
plt.savefig(
    'hyperparameter_impact_plot.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# Display the numerical values used in the plot
hyperparameter_results_df = pd.DataFrame({
    'Number of Estimators': estimators_options,
    'Training Accuracy': training_accuracy_scores,
    'Validation Accuracy': validation_accuracy_scores
})

print("Hyperparameter impact results:")
display(hyperparameter_results_df)

# 2. Model Comparison Matrix

# Compile the evaluation metrics for all three models
classifier_labels = [
    'Decision Tree',
    'Gradient Boosting',
    'SVM'
]

accuracy_values = [
    tree_accuracy,
    gbm_accuracy,
    svm_accuracy
]

f1_score_values = [
    tree_f1_score,
    gbm_f1_score,
    svm_f1_score
]

roc_auc_values = [
    tree_roc_auc,
    gbm_roc_auc,
    svm_roc_auc
]

# Create a performance summary DataFrame
performance_summary_df = pd.DataFrame({
    'Model': classifier_labels,
    'Accuracy': accuracy_values,
    'F1-Score': f1_score_values,
    'ROC-AUC Score': roc_auc_values
})

print("Model performance summary:")
display(performance_summary_df)

# Convert the DataFrame into long format for grouped plotting
melted_performance_df = performance_summary_df.melt(
    id_vars='Model',
    var_name='Performance Metric',
    value_name='Score'
)

# Generate the grouped bar chart
plt.figure(figsize=(12, 7))

comparison_chart = sns.barplot(
    data=melted_performance_df,
    x='Model',
    y='Score',
    hue='Performance Metric',
    palette='Spectral'
)

plt.title(
    'Comparative Performance of the Three Classification Models',
    fontsize=14,
    fontweight='bold'
)

plt.xlabel('Classification Model', fontsize=12)
plt.ylabel('Metric Score', fontsize=12)
plt.ylim(0.80, 1.02)
plt.grid(axis='y', linestyle=':', alpha=0.7)

plt.legend(
    title='Evaluation Metric',
    bbox_to_anchor=(1.01, 1),
    loc='upper left'
)

# Add metric values above the bars
for bar in comparison_chart.patches:
    bar_height = bar.get_height()

    if not np.isnan(bar_height):
        comparison_chart.annotate(
            f'{bar_height:.3f}',
            (
                bar.get_x() + bar.get_width() / 2,
                bar_height
            ),
            ha='center',
            va='bottom',
            fontsize=9,
            xytext=(0, 3),
            textcoords='offset points'
        )

plt.tight_layout()

# Save the plot before displaying it
plt.savefig(
    'model_comparison_matrix.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# 3. Confusion Matrix for the Optimized Ensemble Model

# Gradient Boosting is the ensemble model required by the assignment.
# The labels are ordered as [1, 0] so malignant class 0 is treated
# as the positive medical condition.
#
# Matrix arrangement:
# [[TN, FP],
#  [FN, TP]]
#
# TN = Benign correctly predicted as benign
# FP = Benign incorrectly predicted as malignant
# FN = Malignant incorrectly predicted as benign
# TP = Malignant correctly predicted as malignant

gbm_confusion_matrix = confusion_matrix(
    test_targets,
    gbm_predictions,
    labels=[1, 0]
)

true_negative, false_positive, false_negative, true_positive = (
    gbm_confusion_matrix.ravel()
)

# Create detailed annotations for every matrix cell
confusion_annotations = np.array([
    [
        f'True Negative (TN)\n{true_negative}',
        f'False Positive (FP)\n{false_positive}'
    ],
    [
        f'False Negative (FN)\n{false_negative}',
        f'True Positive (TP)\n{true_positive}'
    ]
])

# Generate the confusion matrix heatmap
plt.figure(figsize=(9, 7))

sns.heatmap(
    gbm_confusion_matrix,
    annot=confusion_annotations,
    fmt='',
    cmap='YlGnBu',
    cbar=True,
    linewidths=0.8,
    linecolor='black',
    xticklabels=[
        'Predicted Benign',
        'Predicted Malignant'
    ],
    yticklabels=[
        'Actual Benign',
        'Actual Malignant'
    ],
    annot_kws={
        'fontsize': 11,
        'fontweight': 'bold'
    }
)

plt.title(
    'Confusion Matrix for Optimized Gradient Boosting Classifier',
    fontsize=14,
    fontweight='bold'
)

plt.xlabel('Predicted Class', fontsize=12)
plt.ylabel('Actual Class', fontsize=12)
plt.tight_layout()

# Save the plot before displaying it
plt.savefig(
    'gradient_boosting_confusion_matrix.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# Display the key error counts
print("Gradient Boosting confusion matrix results:")
print(f"True Negatives: {true_negative}")
print(f"False Positives: {false_positive}")
print(f"False Negatives: {false_negative}")
print(f"True Positives: {true_positive}")

print(
    "\nFalse Negative interpretation: "
    "A malignant tumour was incorrectly classified as benign."
)

print(
    "False Positive interpretation: "
    "A benign tumour was incorrectly classified as malignant."
)
